In [80]:
import gc
import os
import sys
from datetime import timedelta, time
from itertools import combinations
from pathlib import Path

import jdatetime
import numpy as np
import pandas as pd

from statsmodels.tsa.stattools import adfuller
from concurrent.futures import ThreadPoolExecutor

In [53]:
# Add src to path to import project modules
sys.path.insert(0, str(Path.cwd().parent / "src"))

from instruments import InstrumentRegistry

# Load the instrument registry from data/instruments.yaml
registry = InstrumentRegistry()

instruments = [inst for inst in registry]

In [54]:
def jalali_daterange(start: str, end: str) -> list[str]:
    """Generate list of Jalali dates from start → end (inclusive).
    
    Args:
        start: Jalali date string (YYYY-MM-DD)
        end: Jalali date string (YYYY-MM-DD)
    
    Returns:
        List of date strings in YYYY-MM-DD format
    """
    sy, sm, sd = (int(x) for x in start.split("-"))
    ey, em, ed = (int(x) for x in end.split("-"))
    cur = jdatetime.date(sy, sm, sd)
    last = jdatetime.date(ey, em, ed)
    
    if cur > last:
        raise ValueError(f"Start date {start} is after end date {end}")
    
    dates = []
    while cur <= last:
        dates.append(f"{cur.year:04d}-{cur.month:02d}-{cur.day:02d}")
        cur += timedelta(days=1)
    
    return dates

In [55]:
# Lookup instrument metadata for a parquet file
def load_with_metadata(file_path: str | Path):
    """Load parquet and enrich with instrument metadata from registry."""
    file_path = Path(file_path)
    
    # Extract ISIN from filename: {isin}_{jalali_date}.parquet
    filename_stem = file_path.stem  # removes .parquet
    isin = filename_stem.split("_")[0]
    
    # Load the dataframe
    df = load_parquet(file_path)
    
    # Lookup instrument in registry
    try:
        inst = registry.by_isin(isin)
        return df, inst
    except KeyError as e:
        print(f"⚠️  Warning: {e}")
        return df, None

In [58]:
MARKET_OPEN = time(12, 0, 0)
MARKET_END  = time(18, 0, 0)

ORDERBOOK_DIR = "../data/orderbooks"

In [59]:
def load_parquet(file_path: str | Path) -> pd.DataFrame:
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    if not file_path.suffix == ".parquet":
        raise ValueError(f"Expected .parquet file, got: {file_path.suffix}")

    try:
        return pd.read_parquet(file_path)
    except Exception as e:
        raise Exception(f"Failed to read {file_path}: {e}")


def _filter_market_hours(df: pd.DataFrame) -> pd.DataFrame:
    """Filter dataframe rows to market hours based on time column."""
    if 'time' not in df.columns:
        return df
    times = pd.to_datetime(df['time'].astype(str), format='%H:%M:%S').dt.time
    mask  = (times >= MARKET_OPEN) & (times <= MARKET_END)
    return df[mask].reset_index(drop=True)


def load_orderbook_parquet(file_path: str | Path) -> pd.DataFrame:
    """Load an orderbook Parquet file and return as DataFrame."""
    df = load_parquet(file_path)
    return _filter_market_hours(df)


def load_trades_parquet(file_path: str | Path) -> pd.DataFrame:
    """Load a trades Parquet file and return as DataFrame."""
    df = load_parquet(file_path)
    return _filter_market_hours(df)

def get_instrument_orderbook_file_path(inst , date):
    filename = f"{inst.isin:12}_{date}.parquet"
    return os.path.join(ORDERBOOK_DIR, filename)

In [60]:
date_range = jalali_daterange("1404-08-01", "1404-12-07")

In [64]:
def load_orderbook_chunks(date_range, registry, chunk_size=10):
    """
    Generator that loads orderbook data chunk by chunk.
    Yields one pandas DataFrame per chunk of dates.
    """
    all_dates = list(date_range)
    chunks    = [all_dates[i:i+chunk_size] for i in range(0, len(all_dates), chunk_size)]

    for chunk_idx, chunk_dates in enumerate(chunks):
        print(f"chunk {chunk_idx+1}/{len(chunks)}: {chunk_dates[0]} → {chunk_dates[-1]}")

        records = []

        for date in chunk_dates:
            for inst in registry:
                path = get_instrument_orderbook_file_path(inst, date)

                if not os.path.exists(path):
                    continue

                df = load_orderbook_parquet(path)

                if df.empty:
                    continue

                cols = ['time']
                for i in range(1, 6):
                    cols += [
                        f'buy_count_{i}',
                        f'buy_volume_{i}',
                        f'buy_price_{i}',
                        f'sell_price_{i}',
                        f'sell_volume_{i}',
                        f'sell_count_{i}',
                    ]

                df = df[cols]
                df['mid']  = (df['buy_price_1'] + df['sell_price_1']) / 2
                df['date'] = date
                df['isin'] = inst.isin
                records.append(df)

        if not records:
            continue

        chunk_df = (
            pd.concat(records, ignore_index=True)
            .set_index(['date', 'isin', 'time'])
            .sort_index()
        )

        yield chunk_df

        # free memory after yield
        del records, chunk_df
        gc.collect()

In [67]:
CHUNK_SIZE = 10
OUT        = "../data/log_ratios"
os.makedirs(OUT, exist_ok=True)

isins     = [inst.isin for inst in registry]
all_pairs = list(combinations(isins, 2))

all_log_ratios = []

In [68]:
for chunk_df in load_orderbook_chunks(date_range, registry, chunk_size=CHUNK_SIZE):

    chunk_ratios = {}

    for isin_a, isin_b in all_pairs:
        pair = f"{isin_a}/{isin_b}"

        try:
            price_a = chunk_df.xs(isin_a, level='isin')['mid']
            price_b = chunk_df.xs(isin_b, level='isin')['mid']
        except KeyError:
            continue

        price_a, price_b = price_a.align(price_b, join='inner')
        mask    = price_a.notna() & price_b.notna()
        price_a = price_a[mask]
        price_b = price_b[mask]

        if len(price_a) == 0:
            continue

        chunk_ratios[pair] = np.log(price_a) - np.log(price_b)

    # build chunk dataframe — rows=(date,time), columns=pairs
    chunk_ratio_df = pd.DataFrame(chunk_ratios)
    chunk_ratio_df.index.names = ['date', 'time']
    all_log_ratios.append(chunk_ratio_df)

    del chunk_ratios, chunk_ratio_df
    gc.collect()

# final concat
log_ratios_df = pd.concat(all_log_ratios).sort_index()

del all_log_ratios
gc.collect()

chunk 1/13: 1404-08-01 → 1404-08-10
chunk 2/13: 1404-08-11 → 1404-08-20
chunk 3/13: 1404-08-21 → 1404-08-30
chunk 4/13: 1404-09-01 → 1404-09-10
chunk 5/13: 1404-09-11 → 1404-09-20
chunk 6/13: 1404-09-21 → 1404-09-30
chunk 7/13: 1404-10-01 → 1404-10-10
chunk 8/13: 1404-10-11 → 1404-10-20
chunk 9/13: 1404-10-21 → 1404-10-30
chunk 10/13: 1404-11-01 → 1404-11-10
chunk 11/13: 1404-11-11 → 1404-11-20
chunk 12/13: 1404-11-21 → 1404-11-30
chunk 13/13: 1404-12-01 → 1404-12-07


0

In [78]:
# log_ratios_df.to_csv("../data/log_ratios.csv")

In [79]:
log_ratios_df.head(10)

IRTKMOFD0001/IRTKROBA0001  IRTKMOFD0001/IRTKZARA0001  \
date       time                                                             
1404-08-03 12:00:00                   1.035967                   1.228275   
           12:00:01                   1.035597                   1.227616   
           12:00:02                   1.035568                   1.227587   
           12:00:03                   1.035987                   1.227578   
           12:00:04                   1.036040                   1.227578   
           12:00:05                   1.036191                   1.227583   
           12:00:06                   1.036186                   1.227578   
           12:00:07                   1.036854                   1.227561   
           12:00:08                   1.036854                   1.227991   
           12:00:09                   1.037310                   1.227970   

                     IRTKMOFD0001/IRTKLOTF0001  IRTKMOFD0001/IRTKGANJ0001  \
date       time                                                             
1404-08-03 12:00:00                  -0.950708                   1.174889   
           12:00:01                  -0.950143                   1.174136   
           12:00:02                  -0.950173                   1.174324   
           12:00:03                  -0.950175                   1.174316   
           12:00:04                  -0.950175                   1.174316   
           12:00:05                  -0.950169                   1.174321   
           12:00:06                  -0.950174                   1.174483   
           12:00:07                  -0.950191                   1.174466   
           12:00:08                  -0.950191                   1.174594   
           12:00:09                  -0.950212                   1.174574   

                     IRTKMOFD0001/IRTKKIAN0001  IRTKMOFD0001/IRTKJAVA0001  \
date       time                                                             
1404-08-03 12:00:00                  -0.559495                   1.994897   
           12:00:01                  -0.560248                   1.994144   
           12:00:02                  -0.560278                   1.994114   
           12:00:03                  -0.560286                   1.994106   
           12:00:04                  -0.560286                   1.994106   
           12:00:05                  -0.560281                   1.994111   
           12:00:06                  -0.560286                   1.994106   
           12:00:07                  -0.560303                   1.993848   
           12:00:08                  -0.560382                   1.993848   
           12:00:09                  -0.560127                   1.994410   

                     IRTKMOFD0001/IRTKTABA0001  IRTKMOFD0001/IRTKZARF0001  \
date       time                                                             
1404-08-03 12:00:00                   1.930643                  -0.456974   
           12:00:01                   1.921634                  -0.457727   
           12:00:02                   1.912903                  -0.457756   
           12:00:03                   1.912895                  -0.457765   
           12:00:04                   1.912895                  -0.457765   
           12:00:05                   1.921661                  -0.457760   
           12:00:06                   1.921031                  -0.457752   
           12:00:07                   1.921014                  -0.457769   
           12:00:08                   1.921014                  -0.457769   
           12:00:09                   1.921464                  -0.457790   

                     IRTKMOFD0001/IRTKALTN0001  IRTKMOFD0001/IRTKDRKS0001  \
date       time                                                             
1404-08-03 12:00:00                   2.150822                   2.737952   
           12:00:01                   2.150513                   2.737172   
           12

In [81]:
def hurst(series, max_lag=100):
    lags = range(2, max_lag)
    tau  = [np.std(np.subtract(series[lag:].values,
                               series[:-lag].values)) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0]

def test_pair(col):
    series = log_ratios_df[col].dropna()
    if len(series) < 100:
        return col, None, None, None

    # ADF
    adf_p = adfuller(series, maxlag=1, autolag=None)[1]

    # Hurst
    H = hurst(series)

    return col, adf_p, H, adf_p < 0.05 and H < 0.5

with ThreadPoolExecutor() as executor:
    results = list(executor.map(test_pair, log_ratios_df.columns))

mean_reversion_df = pd.DataFrame(
    [(col, p, H, mr) for col, p, H, mr in results if p is not None],
    columns=['pair', 'adf_p', 'hurst', 'mean_reverting']
).set_index('pair')

# valid pairs for trading
valid_pairs = mean_reversion_df[mean_reversion_df['mean_reverting']].index.tolist()

print(mean_reversion_df.sort_values('hurst'))
print(f"\nmean reverting: {len(valid_pairs)} / {len(mean_reversion_df)}")


                                  adf_p     hurst  mean_reverting
pair                                                             
IRTKZFAM0001/IRTKZOMR0001  0.000000e+00  0.312651            True
IRTKATSH0001/IRTKZOMR0001  0.000000e+00  0.325391            True
IRTKZOMR0001/IRTKROSE0001  8.124739e-30  0.325454            True
IRTKZARV0001/IRTKZFAM0001  0.000000e+00  0.326054            True
IRTKZFAM0001/IRTKROSE0001  0.000000e+00  0.327836            True
...                                 ...       ...             ...
IRTKMOFD0001/IRTKLOTF0001  2.029719e-25  0.481408            True
IRTKMOFD0001/IRTKZARF0001  2.989058e-17  0.485079            True
IRTKMOFD0001/IRTKGANJ0001  6.139620e-29  0.485979            True
IRTKMOFD0001/IRTKRITO0001  4.590576e-05  0.507914           False
IRTKMOFD0001/IRTKZARA0001  1.029740e-22  0.508233           False

[136 rows x 3 columns]

mean reverting: 134 / 136


In [82]:
mean_reversion_df.to_csv("../data/mean_reversion_df.csv")